# 🧩 Mini-Lab: JSON Extraction

**Module 4: Prompt Engineering & Security** | **Duration: ~30 min** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why structured output is essential for real-world LLM applications
2. **Use** JSON mode to guarantee valid JSON responses from the API
3. **Parse** LLM output into Python data structures for downstream use
4. **Control** output formatting through prompt instructions and Pydantic validation

## Target Concepts

| Concept | Description |
|---------|-------------|
| Structured Output | Getting formatted, predictable responses from LLMs instead of free-form text |
| JSON Mode | An API setting that guarantees the model returns valid JSON |
| Output Parsing | Extracting and converting LLM text output into usable Python data structures |
| Output Formatting | Controlling the shape and format of the model's response via prompt instructions |

## Setup

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()  # uses OPENAI_API_KEY from .env
MODEL = "gpt-4o-mini"

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## 1. The Problem: Free-Form Text Is Hard to Use

When you ask an LLM a question normally, you get free-form text. That's fine for a chatbot, but **applications** need predictable structure — dictionaries, lists, typed fields — so downstream code can process the result.

Let's see the problem first.

In [2]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Extract the name, age, and city from this text: 'Maria Santos, 34, lives in Lisbon and works as a data engineer.'"}
    ]
)

raw_text = response.choices[0].message.content
md(f"**Raw LLM output:**\n\n{raw_text}")

**Raw LLM output:**

Name: Maria Santos  
Age: 34  
City: Lisbon  

In [3]:
# Try to use this output programmatically — it's just a string!
print(f"Type: {type(raw_text)}")
print(f"Can we access 'name'? No — it's unstructured text.")

Type: <class 'str'>
Can we access 'name'? No — it's unstructured text.


The output is human-readable but not machine-readable. We need **structured output**.

## 2. Output Formatting via Prompt Instructions

The simplest approach: **tell the model what format you want** in the prompt itself. This is *output formatting* — using prompt instructions to control the shape of the response.

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You extract information and return it as JSON. Return ONLY valid JSON, no extra text."},
        {"role": "user", "content": (
            "Extract the name, age, and city from this text:\n\n"
            "'Maria Santos, 34, lives in Lisbon and works as a data engineer.'\n\n"
            "Return JSON with keys: name, age, city"
        )}
    ]
)

formatted_text = response.choices[0].message.content
md(f"**Formatted output:**\n```json\n{formatted_text}\n```")

**Formatted output:**
```json
{
  "name": "Maria Santos",
  "age": 34,
  "city": "Lisbon"
}
```

This often works, but it's **not guaranteed**. The model might add explanation text, markdown fences, or produce invalid JSON. Let's fix that.

## 3. JSON Mode: Guaranteed Valid JSON

OpenAI's `response_format={"type": "json_object"}` forces the model to return **valid JSON**. No extra text, no broken syntax.

> **Important:** When using JSON mode, you **must** mention "JSON" somewhere in your prompt — the API requires it.

In [5]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You extract information and return it as JSON."},
        {"role": "user", "content": (
            "Extract the name, age, and city from this text:\n\n"
            "'Maria Santos, 34, lives in Lisbon and works as a data engineer.'\n\n"
            "Return JSON with keys: name, age, city"
        )}
    ],
    response_format={"type": "json_object"}  # <-- JSON mode!
)

json_text = response.choices[0].message.content
print(f"Raw string: {json_text}")
print(f"Type: {type(json_text)}")

Raw string: {
  "name": "Maria Santos",
  "age": 34,
  "city": "Lisbon"
}
Type: <class 'str'>


The output is a valid JSON **string**. Now we can safely parse it.

## 4. Output Parsing: From JSON String to Python Dict

JSON mode gives us a valid JSON string. **Output parsing** is the step where we convert that string into a Python data structure we can actually use.

In [6]:
# Parse the JSON string into a Python dictionary
parsed = json.loads(json_text)

print(f"Type after parsing: {type(parsed)}")
print(f"Name: {parsed['name']}")
print(f"Age:  {parsed['age']}")
print(f"City: {parsed['city']}")

Type after parsing: <class 'dict'>
Name: Maria Santos
Age:  34
City: Lisbon


Now we have real Python values we can use in application logic — store in a database, pass to another function, or display in a UI.

## 5. Controlling the Schema with Output Formatting

JSON mode guarantees *valid* JSON, but it doesn't guarantee a specific **schema**. The model decides what keys to use. To control the exact structure, we specify the schema in the prompt.

In [7]:
schema_prompt = """Extract event information from the text below.

Return JSON matching this exact schema:
{
  "event_name": "string",
  "date": "YYYY-MM-DD",
  "location": "string",
  "attendees": ["string", ...]
}

Text: "The AI Summit will be held on March 15, 2025 in San Francisco. 
Confirmed speakers include Dr. Sarah Chen, Prof. James Miller, and CEO Ana Ruiz."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You extract structured data and return JSON."},
        {"role": "user", "content": schema_prompt}
    ],
    response_format={"type": "json_object"}
)

event = json.loads(response.choices[0].message.content)
md(f"""**Extracted Event:**\n
| Field | Value |\n|-------|-------|\n"""
   + f"| Event | {event.get('event_name', 'N/A')} |\n"
   + f"| Date | {event.get('date', 'N/A')} |\n"
   + f"| Location | {event.get('location', 'N/A')} |\n"
   + f"| Attendees | {', '.join(event.get('attendees', []))} |")

**Extracted Event:**

| Field | Value |
|-------|-------|
| Event | The AI Summit |
| Date | 2025-03-15 |
| Location | San Francisco |
| Attendees | Dr. Sarah Chen, Prof. James Miller, CEO Ana Ruiz |

By specifying the schema in the prompt, we control **which keys appear, what types they have, and how values are formatted** (e.g., date as `YYYY-MM-DD`).

## 6. Robust Parsing with Pydantic Validation

`json.loads()` gives us a dictionary, but it doesn't check that the schema is correct. **Pydantic** lets us define an expected model and validate + type-check the output automatically.

In [8]:
from pydantic import BaseModel
from typing import List


class Event(BaseModel):
    event_name: str
    date: str
    location: str
    attendees: List[str]


# Validate the parsed dict against our schema
validated_event = Event(**event)

print(f"Type: {type(validated_event)}")
print(f"Event name: {validated_event.event_name}")
print(f"Date: {validated_event.date}")
print(f"Attendees: {validated_event.attendees}")
print(f"Number of attendees: {len(validated_event.attendees)}")

Type: <class '__main__.Event'>
Event name: The AI Summit
Date: 2025-03-15
Attendees: ['Dr. Sarah Chen', 'Prof. James Miller', 'CEO Ana Ruiz']
Number of attendees: 3


Pydantic catches missing fields, wrong types, and other issues — giving you **type-safe** structured data from LLM output.

## 7. Putting It Together: A Reusable Extraction Function

Let's combine everything into a clean pattern: **JSON mode + schema prompt + Pydantic parsing**.

In [9]:
class ContactInfo(BaseModel):
    name: str
    email: str
    company: str
    role: str


def extract_contact(text: str) -> ContactInfo:
    """Extract contact information from unstructured text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You extract contact information and return JSON."},
            {"role": "user", "content": (
                f"Extract contact info from this text:\n\n{text}\n\n"
                "Return JSON with keys: name, email, company, role"
            )}
        ],
        response_format={"type": "json_object"}
    )
    parsed = json.loads(response.choices[0].message.content)
    return ContactInfo(**parsed)

In [10]:
# Test with sample text
sample = (
    "Hi, I'm Alex Kim from NovaTech. I lead the ML platform team. "
    "You can reach me at alex.kim@novatech.io."
)

contact = extract_contact(sample)

md(f"""**Extracted Contact:**

| Field | Value |
|-------|-------|
| Name | {contact.name} |
| Email | {contact.email} |
| Company | {contact.company} |
| Role | {contact.role} |""")

**Extracted Contact:**

| Field | Value |
|-------|-------|
| Name | Alex Kim |
| Email | alex.kim@novatech.io |
| Company | NovaTech |
| Role | Lead, ML Platform Team |

This pattern — **JSON mode + prompt schema + Pydantic validation** — is the standard approach for extracting structured data from LLMs in production applications.

## 🎯 Summary

### Key Takeaways

1. **Structured output** — LLM applications need predictable data formats, not free-form text, so downstream code can process results reliably
2. **JSON mode** — setting `response_format={"type": "json_object"}` guarantees valid JSON output from the API (remember to mention "JSON" in your prompt)
3. **Output formatting** — specifying an exact schema in the prompt controls which keys, types, and value formats the model returns
4. **Output parsing** — `json.loads()` converts the JSON string to a Python dict, and Pydantic adds type-safe validation on top
5. **Production pattern** — combining JSON mode + schema prompts + Pydantic gives you reliable, validated structured data extraction